In [2]:
#question2
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,udf
from pyspark.sql.types import StringType



In [3]:
spark = SparkSession.builder.appName("case-study").getOrCreate()
customers=spark.read \
    .option("header", True) \
    .csv("data/customers.csv")

order_items=spark.read \
    .option("header", True) \
    .csv("data/order_items.csv")
orders=spark.read \
    .option("header", True) \
    .csv("data/orders.csv")
products=spark.read \
    .option("header", True) \
    .csv("data/products.csv")
returns=spark.read \
    .option("header", True) \
    .csv("data/returns.csv")

customers.createOrReplaceTempView("c")
order_items.createOrReplaceTempView("oi")
orders.createOrReplaceTempView("o")
products.createOrReplaceTempView("p")
returns.createOrReplaceTempView("r")

print("total no. of customers",customers.count())
print("total no. of products",products.count())
print("total no. of orders",orders.count())
print("total no. of returned ",returns.count())

customers.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output/q1")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/16 04:03:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


total no. of customers 100000
total no. of products 50000


total no. of orders 1000012
total no. of returned  100000


26/06/16 04:03:54 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
                                                                                

In [4]:
#question 2
sql_df = spark.sql("""
select category ,
sum(unit_cost) AS total_sales
from p
group by category
""")
sql_df.show()

sql_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output/q2")

+--------------+------------------+
|      category|       total_sales|
+--------------+------------------+
|Home & Kitchen| 2901364.330000004|
|        Sports| 2853163.040000003|
|   Electronics|2864604.7399999946|
|      Clothing| 2841424.610000002|
|         Books|2853871.8500000075|
|        Beauty|2919388.7500000037|
|          Toys|2851913.1100000013|
+--------------+------------------+



In [5]:
#question3
q3_df = spark.sql("""
SELECT
    o.customer_id,
    SUM(oi.quantity * oi.selling_price) AS total_purchase
FROM o
JOIN oi
    ON o.order_id = oi.order_id
JOIN p
    ON oi.product_id = p.product_id
GROUP BY o.customer_id
ORDER BY total_purchase DESC
LIMIT 10
""")

q3_df.show()

q3_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output/q3")

26/06/16 04:04:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 04:04:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
                                                                                

+-----------+------------------+
|customer_id|    total_purchase|
+-----------+------------------+
|      93094|181569.68000000005|
|      64560|         164579.62|
|      52275|153364.78999999998|
|      61218|         153067.55|
|      82593|         148281.04|
|      40442|148141.88000000003|
|      20648|148084.28999999998|
|      36102|          147833.9|
|      84830|          147688.4|
|      23289|         146720.19|
+-----------+------------------+



26/06/16 04:04:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 04:04:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
                                                                                

In [25]:
#question 4
q4_df = spark.sql("""
SELECT
    MONTH(o.order_date) AS month,
    CAST(SUM(oi.quantity * oi.selling_price)  
    AS DECIMAL(18,2)) 
    AS total_sales
FROM o
JOIN oi
    ON o.order_id = oi.order_id
JOIN p
    ON oi.product_id = p.product_id
WHERE YEAR(o.order_date) = (
    SELECT MAX(YEAR(order_date))
    FROM o
)
GROUP BY MONTH(o.order_date)
ORDER BY month
""")

q4_df.show()

q4_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output/q4")


26/06/16 04:37:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 04:37:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
                                                                                

+-----+------------+
|month| total_sales|
+-----+------------+
|    1|432531735.41|
|    2|404890345.07|
|    3|432144023.95|
|    4|417131139.67|
|    5|432920484.82|
|    6|420107070.41|
|    7|431797289.97|
|    8|430217700.33|
|    9|420263688.68|
|   10|430186623.01|
|   11|422543689.88|
|   12|431327482.15|
+-----+------------+



26/06/16 04:37:42 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 04:37:42 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
                                                                                

In [26]:
#question5
q5_df = spark.sql("""
SELECT
    p.category,
    ROUND(
        COUNT(DISTINCT r.order_id) * 100.0 /
        COUNT(DISTINCT o.order_id),
        2
    ) AS return_percentage
FROM o
JOIN oi
    ON o.order_id = oi.order_id
JOIN p
    ON oi.product_id = p.product_id
LEFT JOIN r
    ON o.order_id = r.order_id
GROUP BY p.category
ORDER BY return_percentage DESC
""")

q5_df.show()
q5_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output/q5")


[Stage 138:>                                                        (0 + 2) / 2]

+--------------+-----------------+
|      category|return_percentage|
+--------------+-----------------+
|          Toys|            10.05|
|Home & Kitchen|            10.03|
|        Sports|            10.03|
|   Electronics|            10.02|
|         Books|            10.02|
|        Beauty|            10.02|
|      Clothing|             9.97|
+--------------+-----------------+



In [33]:
#question6
q6_df =spark.sql("""
with payment_count as(
select 
    c.state,
    o.payment_mode,
    count(*) as total_orders
from c 
join o
    on c.customer_id = o.customer_id
group by 
c.state,
o.payment_mode
)
select
    state,
    payment_mode,
    total_orders
from(
    select *,
        row_number() over(
        partition by state
        order by total_orders desc
        ) as rn
        from payment_count
)
where rn = 1
order by total_orders DESC
""")


q6_df.show()

q6_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output/q6")

+-----+----------------+------------+
|state|    payment_mode|total_orders|
+-----+----------------+------------+
|   IL|Cash on Delivery|       20204|
|   MI|      Debit Card|       20137|
|   NY|      Debit Card|       20106|
|   OH|     Net Banking|       20068|
|   CA|             UPI|       19990|
|   WA|Cash on Delivery|       19989|
|   TX|             UPI|       19810|
|   GA|     Net Banking|       19784|
|   FL|      Debit Card|       19746|
|   NC|     Net Banking|       19323|
+-----+----------------+------------+

